# Ledgion — Docling parse on Colab (Phase 8, part 1)

Runs **Docling once on a GPU runtime** to pre-parse the FinanceBench filings, and
writes a portable artifact — one JSONL per document plus a run manifest — to
Google Drive. Nothing here runs on the laptop; every downstream step reads the
artifact locally, with no Docling dependency.

**Before running**
1. Runtime → Change runtime type → **GPU** (a T4 is plenty; OCR is off).
2. Put the 5 filing PDFs in a Drive folder (default `MyDrive/ledgion/pdfs`).
3. The repo branch cloned below must already contain
   `ledgion/ingest/parse_docling.py` — commit and push it first, then set
   `REPO_BRANCH` to that branch.

**Resumable:** any document whose `.docling.jsonl` already exists in the output
folder is skipped, so a disconnect costs at most the document in flight.

### 1. Install the pinned Docling

In [ ]:
# Same pin as the `docling` dependency group in pyproject.toml, so the version the
# manifest records matches the version the project declares. Docling brings torch +
# the layout/TableFormer model stack (multi-GB) — fine on Colab, never on the laptop.
!pip install -q docling==2.130.0

### 2. Clone the repo and import the parser (module only — no heavy install)

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/RPreethamR/ledgion.git"
REPO_BRANCH = "main"                 # must contain ledgion/ingest/parse_docling.py
REPO_DIR = Path("/content/ledgion")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

# Add the repo root to sys.path and import the parser MODULE ONLY. We deliberately
# do NOT `pip install` the package — that would pull torch/qdrant/genai and the whole
# local stack. parse_docling.py and the package __init__ files import nothing heavy,
# so this import needs only docling (installed above).
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from ledgion.ingest.parse_docling import (
    ParseConfig,
    build_converter,
    environment_manifest,
    parse_document,
)

print("parser imported from", REPO_DIR)

### 3. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

### 4. Configure paths and list the input PDFs

In [ ]:
from pathlib import Path

# Drive layout — edit these two to match your folders.
INPUT_DIR = Path("/content/drive/MyDrive/ledgion/pdfs")           # the source filing PDFs
OUTPUT_DIR = Path("/content/drive/MyDrive/ledgion/docling_out")   # JSONL + manifest land here
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# OCR off (filings are text-native, verified in the spike); TableFormer accurate.
CONFIG = ParseConfig(use_gpu=True, do_ocr=False, table_mode="accurate")

pdfs = sorted(INPUT_DIR.glob("*.pdf"))
assert pdfs, f"No PDFs found in {INPUT_DIR}"
print(f"{len(pdfs)} PDF(s) to parse:")
for p in pdfs:
    print("  ", p.name)

### 5. Build the converter once (loads the models once for the whole run)

In [ ]:
import time

env = environment_manifest(CONFIG)
print(
    "docling", env["docling_version"],
    "| device:", env["parse_options"]["device"],
    "| gpu:", env["gpu"]["name"],
)

t0 = time.perf_counter()
converter = build_converter(CONFIG)   # loads the layout + TableFormer models once
print(f"converter ready in {time.perf_counter() - t0:.1f}s")

### 6. Parse every PDF (resumable — skips documents already done)

In [ ]:
import json


def _stats_from_jsonl(path: Path, doc_id: str) -> dict:
    """Recover per-document stats from an existing JSONL (resume with no sidecar)."""
    n_elements = n_multi = 0
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            if not line.strip():
                continue
            rec = json.loads(line)
            n_elements += 1
            if rec.get("multi_page"):
                n_multi += 1
    return {
        "doc_id": doc_id,
        "output": path.name,
        "page_count": None,
        "n_elements": n_elements,
        "n_multi_page_elements": n_multi,
        "status": "recovered_from_jsonl",
        "parse_time_s": None,
    }


for pdf in pdfs:
    doc_id = pdf.stem
    out_path = OUTPUT_DIR / f"{doc_id}.docling.jsonl"
    side_path = OUTPUT_DIR / f"{doc_id}.docling.json"

    if out_path.exists():
        # Parsed in a prior (possibly disconnected) session. parse_document writes
        # atomically, so an existing file is always complete — skip the costly parse.
        # Backfill the per-doc sidecar if it went missing, so the manifest stays whole.
        if not side_path.exists():
            side_path.write_text(
                json.dumps(_stats_from_jsonl(out_path, doc_id), indent=2), encoding="utf-8"
            )
        print(f"skip  {doc_id}  (output exists)")
        continue

    print(f"parse {doc_id} ...", end=" ", flush=True)
    stats = parse_document(converter, pdf, out_path, doc_id=doc_id)
    side_path.write_text(json.dumps(stats, indent=2), encoding="utf-8")
    print(
        f"{stats['n_elements']} elements, "
        f"{stats['n_multi_page_elements']} multi-page, "
        f"{stats['page_count']} pages, {stats['parse_time_s']}s"
    )

### 7. Write the run manifest back to Drive

In [ ]:
import datetime
import json

# Assemble from the per-document sidecars so the manifest is complete even across
# resumes/disconnects — every parsed document left one.
docs = [
    json.loads(side.read_text(encoding="utf-8"))
    for side in sorted(OUTPUT_DIR.glob("*.docling.json"))
]

manifest = environment_manifest(CONFIG)
manifest["generated_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
manifest["repo_branch"] = REPO_BRANCH
manifest["input_dir"] = str(INPUT_DIR)
manifest["output_dir"] = str(OUTPUT_DIR)
manifest["documents"] = docs
manifest["totals"] = {
    "documents": len(docs),
    "elements": sum(d.get("n_elements") or 0 for d in docs),
    "multi_page_elements": sum(d.get("n_multi_page_elements") or 0 for d in docs),
    "pages": sum(d.get("page_count") or 0 for d in docs),
}

manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("wrote", manifest_path)
print(json.dumps(manifest["totals"], indent=2))

## Outputs (in the Drive output folder)

- `<doc_id>.docling.jsonl` — one JSON element per line, in reading order.
- `<doc_id>.docling.json` — per-document stats (also folded into the manifest).
- `manifest.json` — Docling/model versions, parse options, GPU, per-document
  timings and page counts, and the corpus-wide multi-page element count.

Download these into the repo's local artifact folder to build the Docling
parser-ablation index locally — no further Colab run needed to try a different
table serialization, because every table's full cell grid is in the JSONL.